In [1]:
import matplotlib.pyplot as plt
import parselib
import sys
import seaborn as sns
import matplotlib as mpl
import plotconfig
from matplotlib.lines import Line2D

In [2]:
args = {
    "chaos": "on_1",
    "timestamp": ["20251119", "202512", "202511", "2025112"],
    "cca": ["cubic","bbr", "bbr2", "bbr3"],
    "test_cca": "yes",
    "kernel": ["kernel6-1", "zkernel5-13-BBRv2", "zkernel6-13-BBRv3"],
    "loss_mode": "none",
    "n": [30],
    "parallel": [1],
    "bdp": [1],
    "rate": [100],
    "delay_rtt": [10,20,30],
    "deadline_run": [1000000, 10000000, 20000000],
}

metric = "bits_per_second"

baselogpath = "../data"

In [3]:
# create filter and populate it from args dictionary
fil = parselib.Filter()
fil.fill_from_dict(args)
df = parselib.logs_to_df(baselogpath, fil)
if df is None:
    sys.exit()

start 14280
end 14280
bytes 14280
bits_per_second 14280
mbps_timeseries 14280
rttms_timeseries 14280
retransmits 14280
timestamp 14280
iteration 14280
cpu_host_total 14280
cpu_host_user 14280
cpu_host_system 14280
cpu_remote_total 14280
chaos 14280
deadline_run 14280
deadline_period 14280
os 14280
bdp 14280
setup 14280
cca 14280
cpus 14280
kernel 14280
mode 14280
loss 14280
rate 14280
delay_rtt 14280
buffer_size_bytes 14280
parallel 14280
socket_buffer 14280
app_buffer 14280
n 14280
sysctl_cmd 14280
vm 14280
bandwidth_delay_product 14280
loss_mode 14280
vms 14280
pacing 14280
hyperthreading 14280
tso 14280
qdisc 14280
hpet 14280
tsc 14280
hostq 14280
loadperc 14280
deadline_period_factor 14280
random_loss_rate 14280
gemodel_q 14280
original_cca 14280
test_cca 14280
default_qdisc 14280
json 14280


In [4]:
df["slice_perc"] = round((df["deadline_run"]/df["deadline_period"])*100)
df["mbps"] = df["bits_per_second"]/1000000

In [5]:
def plot(df, savefig = False):    
    if savefig:
        mpl.use('agg')
    plotconfig.configure_conext()
    width = plotconfig.pt2inch(plotconfig.COLUMN_WIDTH)
    height = width*(1/3)
    FIG_SIZE = (width, height)
    
    paletti = plotconfig.COLORS[:3]
    fig, axs = plt.subplots(1,3,figsize=FIG_SIZE, sharey=True,constrained_layout=True)


    for it, cca in enumerate(sorted(df["cca"].unique())):
        if cca == "cubic":
            df_cubic = df[df["cca"] == "cubic"]
            df_cubic = df_cubic[df_cubic["kernel"] == "kernel6-1"]
            df_cubic = df_cubic.sort_values(by=['slice_perc', "delay_rtt"])
            df_cubic = df_cubic[["slice_perc", "mbps","deadline_run", "delay_rtt"]].groupby(["slice_perc", "deadline_run", "delay_rtt"],as_index=False).median()
            for bbr_it in [0,1,2]:
                for rtt_ in df_cubic["delay_rtt"].unique():
                    if rtt_ == 10:
                        marker = "o"
                    elif rtt_ == 20:
                        marker = "v"
                    elif rtt_ == 30:
                        marker = "s"
                    elif rtt_ == 40:
                        marker = "P"
                    sns.stripplot(ax=axs[bbr_it], data=df_cubic[df_cubic["delay_rtt"] == rtt_], x="slice_perc", y="mbps", hue="deadline_run", legend=False, dodge=True, jitter=True, alpha=0.9, palette=paletti, zorder=0, marker=marker, size=1.75, edgecolor='black',linewidth=0.5)
                
                if bbr_it != 1:
                    axs[bbr_it].set_xlabel("")
            continue

        data_sorted=df[df["cca"] == cca].sort_values(by=['slice_perc', 'delay_rtt'])

        extra_legend_patches = []
        
        for rtt_ in data_sorted["delay_rtt"].unique():
            if rtt_ == 10:
                marker = "o"
            elif rtt_ == 20:
                marker = "v"
            elif rtt_ == 30:
                marker = "s"
            elif rtt_ == 40:
                marker = "P"
            extra_legend_patches.append(Line2D([0], [0], linestyle='none', mfc=paletti[2], markersize=3,mec=paletti[2], marker=marker, label=f'{int(rtt_)}ms'))
            data_ = data_sorted[["slice_perc", "mbps","deadline_run", "delay_rtt"]].groupby(["slice_perc", "deadline_run", "delay_rtt"],as_index=False).median()
            sns.stripplot(ax=axs[it], data=data_[data_["delay_rtt"] == rtt_], x="slice_perc", y="mbps", hue="deadline_run", jitter=False, alpha=0.6, palette=paletti, zorder=0,dodge=True, legend=True if rtt_ == 10 else False, marker = marker, size=3,linewidth=0.1)

        axs[it].set_xticks([0,1,2,3,4,5,6,7,8,9,10,11,12], ["10","","20","","30","","40","","50","","60","","70"], fontsize=plotconfig.FONT_SIZE-2)
        axs[it].tick_params(axis='both', which='major', labelsize=plotconfig.FONT_SIZE-2)
        axs[it].vlines([0.5,1.5,2.5,3.5,4.5,5.5,6.5,7.5,8.5,9.5,10.5,11.5], ymin=-5, ymax = int(args["rate"][0])+5, color="gainsboro", linewidth=0.5)
        axs[it].set_xlim(-0.5, 12.5)
    
        if cca == "bbr":
            axs[it].set_title("BBRv1",fontsize=plotconfig.FONT_SIZE-2,pad=3)
            axs[it].set_xlabel("")
            axs[it].legend([])
        elif cca == "bbr2":
            axs[it].set_title("BBRv2",fontsize=plotconfig.FONT_SIZE-2,pad=3)
            axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-2)
            axs[it].legend([])
        elif cca == "bbr3":
            axs[it].set_title("BBRv3",fontsize=plotconfig.FONT_SIZE-2,pad=3)
            axs[it].set_xlabel("")
            han = [
                Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.1, marker="o", markersize=3, label='BBRvx'),
                Line2D([0], [0], linestyle='none', mfc=paletti[2], mec="black", mew=0.6, marker="o", markersize=1.8, label='Cubic'),
            ]
            leg1 = axs[it].legend(handles = han,loc="lower right", bbox_to_anchor=(1.3, 0.6), handlelength=1, framealpha=0.4,title_fontsize=plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3)
            axs[it].add_artist(leg1)
            h, l = axs[it].get_legend_handles_labels()
            for enu,hii in enumerate(h):
                hii.set_alpha(1)
                l[enu] = f"{int(int(l[enu])/1000000)}ms"
            leg2 = axs[it].legend(handles=h, labels= l, title="timeslice", loc="lower right", handlelength=1.7, bbox_to_anchor=(1.3,0.3), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)
            axs[it].add_artist(leg2)
            axs[it].legend(handles=extra_legend_patches, title="RTT", loc="lower right", handlelength=1.7, bbox_to_anchor=(1.3, 0), title_fontsize = plotconfig.FONT_SIZE-3, fontsize=plotconfig.FONT_SIZE-3, framealpha=0.4)

    axs[0].set_ylabel("Avg. throughput [Mbps]",fontsize=plotconfig.FONT_SIZE-2)
    axs[0].set_ylim(-float(args["rate"][0])*0.05,int(args["rate"][0]))
    
    if savefig:
        fig.savefig(f"figures/figure_4.pdf", format="pdf")
    else:
        plt.show()

<>:63: SyntaxWarning: invalid escape sequence '\%'
<>:63: SyntaxWarning: invalid escape sequence '\%'
/tmp/ipykernel_217423/603211451.py:63: SyntaxWarning: invalid escape sequence '\%'
  axs[it].set_xlabel("VM CPU share [\%]",fontsize=plotconfig.FONT_SIZE-2)


In [6]:
plot(df, True)